# Human, Synthetic, or Both? — End-to-End Pipeline Demo

This notebook walks through the full pipeline used in the study, end to end:
collect human data → generate synthetic rewrites → build a hybrid dataset →
fine-tune four Pythia-70M models → evaluate perplexity, ARC-Easy, and HellaSwag.

It's meant for quick exploration on a Colab GPU. For production-style, reusable
code, see the equivalent scripts in [`src/`](../src/) — this notebook calls the
same logic inline.

See the [project README](../README.md) for results and the
[full paper](../paper/human_synthetic_or_both.pdf) for the complete write-up.


## 0. Setup

In [ ]:
!pip install -q torch transformers>=4.40.0 datasets accelerate sentencepiece

## 1. Load and clean human-written data

Sample paragraphs from Wikipedia, filter by length, and split into train/val.


In [ ]:
from datasets import load_dataset
import os

def _normalize_text_field(row):
    t = row.get("text", "")
    if isinstance(t, dict):
        t = t.get("text", "")
    return str(t) if t is not None else ""

def load_human_any(max_paragraphs=12000, fraction="1%"):
    paras = []
    try:
        ds = load_dataset("wikimedia/wikipedia", "20231101.en", split=f"train[:{fraction}]")
        for ex in ds:
            txt = _normalize_text_field(ex)
            if not txt:
                continue
            for p in txt.split("\n"):
                p = p.strip()
                w = p.split()
                if 10 <= len(w) <= 500:
                    paras.append(p)
                    if len(paras) >= max_paragraphs:
                        return paras
        return paras
    except Exception as e:
        print("wikimedia/wikipedia failed:", type(e).__name__)
        ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:10%]")
        for ex in ds:
            p = _normalize_text_field(ex).strip()
            if p and not p.startswith(" =") and len(p.split()) >= 10:
                paras.append(p)
                if len(paras) >= max_paragraphs:
                    return paras
        return paras

paras = load_human_any(max_paragraphs=12000, fraction="1%")
print("total paragraphs:", len(paras))

os.makedirs("data", exist_ok=True)
split = int(0.9 * len(paras))
train_paras, val_paras = paras[:split], paras[split:]

with open("data/human_train.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_paras) + "\n")
with open("data/human_val.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(val_paras) + "\n")

print("saved train:", len(train_paras), "val:", len(val_paras))

## 2. Generate synthetic data — Flan-T5-Large

Rewrite every human paragraph into a more concise, information-dense version
using a seq2seq summarization prompt.


In [ ]:
from transformers import pipeline
import itertools

IN_FILE  = "data/human_train.txt"
OUT_FILE = "data/synthetic_train_flant5.txt"
MODEL    = "google/flan-t5-large"
MAX_NEW  = 160
BATCH    = 8
LIMIT    = None  # set e.g. 20 for a quick smoke test

def read_lines(p, limit=None):
    with open(p, encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    return lines if limit is None else lines[:limit]

def write_lines(p, lines):
    with open(p, "w", encoding="utf-8") as f:
        for x in lines:
            f.write(x.replace("\n", " ") + "\n")

def batched(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            break
        yield chunk

src = read_lines(IN_FILE, LIMIT)
print("loaded:", len(src))

gen = pipeline("text2text-generation", model=MODEL, device_map="auto")

out, processed = [], 0
for chunk in batched(src, BATCH):
    prompts = [
        "Summarize the paragraph below into 2-3 sentences using different wording. "
        "Make it more information-dense and do not copy full sentences verbatim. "
        "Preserve all key facts:\n\n" + p
        for p in chunk
    ]
    preds = gen(prompts, max_new_tokens=MAX_NEW, batch_size=BATCH, truncation=True,
                do_sample=True, temperature=0.8, top_p=0.9)
    for i, pred in enumerate(preds):
        text = pred.get("generated_text") if isinstance(pred, dict) else pred[0].get("generated_text")
        if not text or len(text.split()) < 3:
            text = chunk[i]
        out.append(text.strip())
    processed += len(chunk)
    if processed % 100 == 0:
        print(f"generated {processed}/{len(src)}")
        write_lines(OUT_FILE, out)

write_lines(OUT_FILE, out)
print("wrote:", OUT_FILE, "lines:", len(out))

## 3. Generate synthetic data — Gemma-2B-it

`google/gemma-2b-it` is gated on Hugging Face — run the login cell once and
accept the model license at https://huggingface.co/google/gemma-2b-it first.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import pipeline
import itertools

IN_FILE  = "data/human_train.txt"
OUT_FILE = "data/synthetic_train_gemma2b.txt"
MODEL    = "google/gemma-2b-it"
MAX_NEW  = 160
BATCH    = 4
LIMIT    = None

def clean_gemma_output(text):
    bad_prefixes = ("sure", "here is", "here's", "below is",
                     "the rewritten paragraph", "rewritten paragraph")
    t = text.strip()
    lower = t.lower()
    for bp in bad_prefixes:
        if lower.startswith(bp):
            t = t.split(".", 1)[-1].strip()
            break
    return t

src = read_lines(IN_FILE, LIMIT)
print("loaded:", len(src))

gen = pipeline("text-generation", model=MODEL, device_map="auto")

out, processed = [], 0
for chunk in batched(src, BATCH):
    prompts = [
        "<start_of_turn>user\n"
        "Rewrite the paragraph below in a neutral, encyclopedic style similar to Wikipedia.\n\n"
        "Requirements:\n"
        "- Output ONLY the rewritten paragraph text.\n"
        "- Do NOT include introductions like \"Sure\" or \"Here is\".\n"
        "- Preserve all factual content exactly.\n"
        "- Use concise, information-dense sentences.\n\n"
        "Paragraph:\n" + p + "\n"
        "<end_of_turn>\n<start_of_turn>model\n"
        for p in chunk
    ]
    preds = gen(prompts, max_new_tokens=MAX_NEW, batch_size=BATCH, truncation=True,
                do_sample=True, temperature=0.8, top_p=0.9,
                pad_token_id=gen.tokenizer.eos_token_id)
    for i, pred in enumerate(preds):
        text = pred[0]["generated_text"].strip()
        if text.startswith(prompts[i]):
            text = text[len(prompts[i]):].strip()
        text = clean_gemma_output(text)
        if not text:
            text = chunk[i]
        out.append(text)
    processed += len(chunk)
    if processed % 100 == 0:
        print(f"generated {processed}/{len(src)}")
        write_lines(OUT_FILE, out)

write_lines(OUT_FILE, out)
print("wrote:", OUT_FILE, "lines:", len(out))

## 4. Build the hybrid dataset

20% human-written paragraphs + 80% Gemma-rewritten paragraphs, shuffled.


In [ ]:
import random

def mix_datasets(human_file, synthetic_file, human_fraction=0.2, seed=42, out_file="data/mix.txt"):
    with open(human_file, encoding="utf-8") as f:
        human = [l.strip() for l in f if l.strip()]
    with open(synthetic_file, encoding="utf-8") as f:
        synth = [l.strip() for l in f if l.strip()]
    n = min(len(human), len(synth))
    k = int(n * human_fraction)
    mixed = human[:k] + synth[k:n]
    random.seed(seed)
    random.shuffle(mixed)
    with open(out_file, "w", encoding="utf-8") as f:
        f.write("\n".join(mixed) + "\n")
    print(f"human: {k} ({human_fraction:.0%})  synthetic: {n - k} ({1 - human_fraction:.0%})  -> {out_file}")

mix_datasets("data/human_train.txt", "data/synthetic_train_gemma2b.txt",
             human_fraction=0.2, out_file="data/mix.txt")

## 5. Train all four models

Same architecture, hyperparameters, and seed for every run — only the
training file changes. Pulled in from `src/train.py` so the notebook and
the importable script stay in sync; run this cell once per dataset.


In [ ]:
import subprocess, sys

runs = [
    ("data/human_train.txt",            "runs/human-70m"),
    ("data/synthetic_train_flant5.txt", "runs/synthetic-70m-flan"),
    ("data/synthetic_train_gemma2b.txt","runs/synthetic-70m"),
    ("data/mix.txt",                    "runs/mix-70m"),
]

for train_file, out_dir in runs:
    print(f"\n=== Training on {train_file} -> {out_dir} ===")
    subprocess.run([
        sys.executable, "../src/train.py",
        "--train_file", train_file,
        "--eval_file", "data/human_val.txt",
        "--out_dir", out_dir,
        "--num_epochs", "2",
        "--lr", "1e-4",
    ], check=True)

## 6. Evaluate — Perplexity (Table 1)

In [ ]:
import sys
sys.path.append("../src/evaluate")
from common import load_local_model, generate_answer
import subprocess
subprocess.run([sys.executable, "../src/evaluate/eval_perplexity.py"], check=True)

## 7. Evaluate — ARC-Easy reasoning (Table 2)

In [ ]:
subprocess.run([sys.executable, "../src/evaluate/eval_arc_easy.py", "--sample_size", "50"], check=True)

## 8. Evaluate — HellaSwag commonsense reasoning (Table 3)

In [ ]:
subprocess.run([sys.executable, "../src/evaluate/eval_hellaswag.py", "--sample_size", "50"], check=True)

## 9. Qualitative comparison

Same prompt, four models — inspect fluency and factuality side by side.

In [ ]:
subprocess.run([sys.executable, "../src/evaluate/eval_qualitative.py", "--num_examples", "5"], check=True)